# Data Preparation

This notebook inspects the datasets under `maxtoki_development_data` and prints a small head/preview for each file.

The data directory is large, so the code below is intentionally conservative:

- `.h5ad` files are opened with `anndata.read_h5ad(..., backed="r")` when `anndata` is available.
- `.mtx.gz` files are previewed by reading the Matrix Market header and first coordinate rows rather than loading the full matrix.
- `.tsv.gz` files are previewed by streaming the first lines and first columns.
- `.loom` files are previewed with `h5py` by reading only a small matrix slice and attribute keys.
- Supporting `.txt` and `.pickle` files are inspected separately at the end.


In [1]:
from pathlib import Path
import gzip
import os
import pickle
import sys
from datetime import datetime

try:
    import pandas as pd
except Exception as exc:
    pd = None
    print(f"pandas unavailable: {exc}")

try:
    import numpy as np
except Exception as exc:
    np = None
    print(f"numpy unavailable: {exc}")

try:
    import anndata as ad
except Exception as exc:
    ad = None
    print(f"anndata unavailable: {exc}")

try:
    import h5py
except Exception as exc:
    h5py = None
    print(f"h5py unavailable: {exc}")

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda text: text

DATA_DIR = Path("/gladstone/theodoris/lab/enockniyonkuru/maxtoki_development_data")
HEAD_N = 5
MAX_COLUMNS = 8

print(f"Notebook run started: {datetime.now().isoformat(timespec='seconds')}")
print(f"Data directory: {DATA_DIR}")
print(f"Exists: {DATA_DIR.exists()}")


Notebook run started: 2026-04-15T16:11:14
Data directory: /gladstone/theodoris/lab/enockniyonkuru/maxtoki_development_data
Exists: True


## Dataset Manifest

This manifest is the current 21-file dataset scope: 10 CellxGene `.h5ad` files, 1 HCA Matrix Market file, 5 UCSC/local matrices, and 5 lab `.loom` files.

In [2]:
DATASETS = [
    {
        "source": "CellxGene",
        "name": "Construction of a human cell landscape at single-cell level",
        "path": DATA_DIR / "source_1_cellxgene/01_construction_of_a_human_cell_landscape_at_single_cell_level.h5ad",
        "origin": "https://datasets.cellxgene.cziscience.com/74c3403a-451c-4a62-84e0-d8a8e45c7ea7.h5ad",
    },
    {
        "source": "CellxGene",
        "name": "Survey of human embryonic development (1 million cells subset)",
        "path": DATA_DIR / "source_1_cellxgene/02_survey_of_human_embryonic_development_1_million_cells_subset.h5ad",
        "origin": "https://datasets.cellxgene.cziscience.com/c3bf819d-423d-435f-b8d0-e36ad6088138.h5ad",
    },
    {
        "source": "CellxGene",
        "name": "Survey of human embryonic development",
        "path": DATA_DIR / "source_1_cellxgene/03_survey_of_human_embryonic_development.h5ad",
        "origin": "https://datasets.cellxgene.cziscience.com/dd2564a6-27a0-433a-893c-72475e4a39fe.h5ad",
    },
    {
        "source": "CellxGene",
        "name": "Sex-Specific Control of Human Heart Maturation by the Progesterone Receptor",
        "path": DATA_DIR / "source_1_cellxgene/04_sex_specific_control_of_human_heart_maturation_by_the_progesterone_receptor.h5ad",
        "origin": "https://datasets.cellxgene.cziscience.com/d61a74ab-e1ef-4ced-9131-698bf7d94be2.h5ad",
    },
    {
        "source": "CellxGene",
        "name": "Integrated adult and foetal hearts",
        "path": DATA_DIR / "source_1_cellxgene/05_integrated_adult_and_foetal_hearts.h5ad",
        "origin": "https://datasets.cellxgene.cziscience.com/51073d23-97b7-4c05-84eb-5a18024e966c.h5ad",
    },
    {
        "source": "CellxGene",
        "name": "Rotem_12W_heart_C1",
        "path": DATA_DIR / "source_1_cellxgene/06_rotem_12w_heart_c1.h5ad",
        "origin": "https://datasets.cellxgene.cziscience.com/380b448b-85de-4953-8e82-2fda20276a12.h5ad",
    },
    {
        "source": "CellxGene",
        "name": "Rotem_12W_heart_B1",
        "path": DATA_DIR / "source_1_cellxgene/07_rotem_12w_heart_b1.h5ad",
        "origin": "https://datasets.cellxgene.cziscience.com/57c7e31e-6bf5-4498-a2d3-2a3728c64ded.h5ad",
    },
    {
        "source": "CellxGene",
        "name": "Rotem_12W_heart_D1",
        "path": DATA_DIR / "source_1_cellxgene/08_rotem_12w_heart_d1.h5ad",
        "origin": "https://datasets.cellxgene.cziscience.com/72e35ce5-fb20-46d0-adf9-a8c7f44af287.h5ad",
    },
    {
        "source": "CellxGene",
        "name": "Rotem_12W_heart_A1",
        "path": DATA_DIR / "source_1_cellxgene/09_rotem_12w_heart_a1.h5ad",
        "origin": "https://datasets.cellxgene.cziscience.com/731fb49f-4a81-48d5-9f28-ee1b08f1018d.h5ad",
    },
    {
        "source": "CellxGene",
        "name": "Single-nuclei RNA-seq of human outflow tract and aortic valve tissue",
        "path": DATA_DIR / "source_1_cellxgene/10_single_nuclei_rna_seq_human_outflow_tract_aortic_valve.h5ad",
        "origin": "https://datasets.cellxgene.cziscience.com/e6c07fbd-c90b-48c0-b6e3-b03b2d7218c5.h5ad",
    },
    {
        "source": "Human Early Embryogenesis Atlas",
        "name": "GSE157329 raw counts",
        "path": DATA_DIR / "source_2_human_early_embryogenesis_atlas/93a9e248-b704-419f-b5ab-a0b96eefeaa0/GSE157329_raw_counts.mtx.gz",
        "origin": "HCA/Azul manifest URL from documents/data_sources.txt",
    },
    {
        "source": "UCSC Cells",
        "name": "Human cardiogenesis in vitro expression matrix",
        "path": DATA_DIR / "source_3_ucsc_cells/01_human_cardiogenesis_in_vitro_exprMatrix.tsv.gz",
        "origin": "Local copy corresponding to https://cells.ucsc.edu/cardiogenesis-atac/in-vitro/exprMatrix.tsv.gz",
    },
    {
        "source": "UCSC Cells",
        "name": "Human cardiogenesis in vivo expression matrix",
        "path": DATA_DIR / "source_3_ucsc_cells/02_human_cardiogenesis_in_vivo_exprMatrix.tsv.gz",
        "origin": "Local copy corresponding to https://cells.ucsc.edu/cardiogenesis-atac/in-vivo/exprMatrix.tsv.gz",
    },
    {
        "source": "UCSC Cells",
        "name": "Multiomic human heart snRNA-seq matrix",
        "path": DATA_DIR / "source_3_ucsc_cells/03_multiomic_human_heart_snrna_seq_matrix.mtx.gz",
        "origin": "Local copy corresponding to https://cells.ucsc.edu/multiomic-human-heart/rna/matrix.mtx.gz",
    },
    {
        "source": "UCSC Cells",
        "name": "Multiomic human heart snATAC-seq matrix",
        "path": DATA_DIR / "source_3_ucsc_cells/04_multiomic_human_heart_snatac_seq_matrix.mtx.gz",
        "origin": "Local copy corresponding to https://cells.ucsc.edu/multiomic-human-heart/atac/matrix.mtx.gz",
    },
    {
        "source": "UCSC Cells",
        "name": "Heart of Cells overall heart scRNA-seq expression matrix",
        "path": DATA_DIR / "source_3_ucsc_cells/05_heart_of_cells_overall_heart_scrna_seq_exprMatrix.tsv.gz",
        "origin": "Local copy corresponding to UCSC Heart of Cells overall heart exprMatrix.tsv.gz",
    },
    {
        "source": "Lab Directory",
        "name": "Human fetal cis-regulatory elements",
        "path": DATA_DIR / "source_4_lab_directory/01_human_fetal_cis_regulatory_elements.loom",
        "origin": "/gladstone/theodoris/lab/genecorpus_XM/corpus_loom_files/genecorpus_95M_loom/9010519993/File_S3_Matrix.Gene_Raw_Counts_of_Cells.loom",
    },
    {
        "source": "Lab Directory",
        "name": "Human fetal striatum atlas",
        "path": DATA_DIR / "source_4_lab_directory/02_human_fetal_striatum_atlas.loom",
        "origin": "/gladstone/theodoris/lab/genecorpus_XM/corpus_loom_files/genecorpus_95M_loom/E-MTAB-8894/E-MTAB-8894.loom",
    },
    {
        "source": "Lab Directory",
        "name": "Human megakaryocyte development: yolk-sac stage",
        "path": DATA_DIR / "source_4_lab_directory/03_human_megakaryocyte_development/GSE144024_YS_Stage_gene.loom",
        "origin": "/gladstone/theodoris/lab/genecorpus_XM/corpus_loom_files/genecorpus_95M_loom/4110056855/",
    },
    {
        "source": "Lab Directory",
        "name": "Human megakaryocyte development: hESC Day 0",
        "path": DATA_DIR / "source_4_lab_directory/03_human_megakaryocyte_development/GSE144024_hESC_Day0_gene.loom",
        "origin": "/gladstone/theodoris/lab/genecorpus_XM/corpus_loom_files/genecorpus_95M_loom/4110056855/",
    },
    {
        "source": "Lab Directory",
        "name": "Fetal vs. adult human epicardium",
        "path": DATA_DIR / "source_4_lab_directory/04_fetal_vs_adult_human_epicardium.loom",
        "origin": "/gladstone/theodoris/lab/genecorpus_XM/corpus_loom_files/genecorpus_95M_loom/cxg/5500c673-1610-40a0-86d9-64d987ae50e6.loom",
    },
]

SUPPORTING_FILES = [
    DATA_DIR / "source_4_lab_directory/03_human_megakaryocyte_development/GSE144024_YS_Stage_gene_log.txt",
    DATA_DIR / "source_4_lab_directory/03_human_megakaryocyte_development/GSE144024_YS_Stage_gene_per_cell_summary.pickle",
    DATA_DIR / "source_4_lab_directory/03_human_megakaryocyte_development/GSE144024_YS_Stage_gene_postcheck.txt",
    DATA_DIR / "source_4_lab_directory/03_human_megakaryocyte_development/GSE144024_hESC_Day0_gene_log.txt",
    DATA_DIR / "source_4_lab_directory/03_human_megakaryocyte_development/GSE144024_hESC_Day0_gene_per_cell_summary.pickle",
    DATA_DIR / "source_4_lab_directory/03_human_megakaryocyte_development/GSE144024_hESC_Day0_gene_postcheck.txt",
]


In [3]:
def format_bytes(num_bytes):
    num_bytes = float(num_bytes)
    units = ["B", "KiB", "MiB", "GiB", "TiB"]
    value = num_bytes
    for unit in units:
        if value < 1024 or unit == units[-1]:
            if unit == "B":
                return f"{int(value)}{unit}"
            return f"{value:.1f}{unit}"
        value /= 1024


def file_size(path):
    path = Path(path)
    return path.stat().st_size if path.exists() else None


def show_table(rows, columns=None):
    if pd is not None:
        display(pd.DataFrame(rows, columns=columns))
    else:
        if columns:
            print("\t".join(map(str, columns)))
        for row in rows:
            print("\t".join(map(str, row)))


manifest_rows = []
for i, ds in enumerate(DATASETS, start=1):
    size = file_size(ds["path"])
    manifest_rows.append([
        i,
        ds["source"],
        ds["name"],
        format_bytes(size) if size is not None else "missing",
        str(ds["path"].relative_to(DATA_DIR)) if ds["path"].exists() else str(ds["path"]),
    ])

show_table(manifest_rows, columns=["#", "source", "dataset", "size", "relative_path"])


,#,source,dataset,size,relative_path
0,1,CellxGene,Construction of a human cell landscape at sing...,1.3GiB,source_1_cellxgene/01_construction_of_a_human_...
1,2,CellxGene,Survey of human embryonic development (1 milli...,4.6GiB,source_1_cellxgene/02_survey_of_human_embryoni...
2,3,CellxGene,Survey of human embryonic development,18.7GiB,source_1_cellxgene/03_survey_of_human_embryoni...
3,4,CellxGene,Sex-Specific Control of Human Heart Maturation...,816.5MiB,source_1_cellxgene/04_sex_specific_control_of_...
4,5,CellxGene,Integrated adult and foetal hearts,409.1MiB,source_1_cellxgene/05_integrated_adult_and_foe...
5,6,CellxGene,Rotem_12W_heart_C1,34.0MiB,source_1_cellxgene/06_rotem_12w_heart_c1.h5ad
6,7,CellxGene,Rotem_12W_heart_B1,36.7MiB,source_1_cellxgene/07_rotem_12w_heart_b1.h5ad
7,8,CellxGene,Rotem_12W_heart_D1,24.2MiB,source_1_cellxgene/08_rotem_12w_heart_d1.h5ad
8,9,CellxGene,Rotem_12W_heart_A1,42.8MiB,source_1_cellxgene/09_rotem_12w_heart_a1.h5ad
9,10,CellxGene,Single-nuclei RNA-seq of human outflow tract a...,437.1MiB,source_1_cellxgene/10_single_nuclei_rna_seq_hu...


## Preview Helpers

These functions print small heads/previews without materializing the full datasets.

In [4]:
def print_dataset_header(ds):
    path = Path(ds["path"])
    display(Markdown(f"### {ds['name']}"))
    print(f"Source: {ds['source']}")
    print(f"Path: {path}")
    print(f"Origin: {ds['origin']}")
    if not path.exists():
        print("MISSING")
        return False
    print(f"Size: {format_bytes(path.stat().st_size)}")
    return True


def preview_delimited_gz(path, n=HEAD_N, max_cols=MAX_COLUMNS, sep="\t"):
    rows = []
    with gzip.open(path, "rt", errors="replace") as handle:
        for idx, line in enumerate(handle):
            if idx >= n:
                break
            parts = line.rstrip("\n").split(sep)
            clipped = parts[:max_cols]
            if len(parts) > max_cols:
                clipped.append(f"... {len(parts) - max_cols} more columns")
            rows.append(clipped)
    show_table(rows)


def preview_mtx_gz(path, n=HEAD_N):
    header = None
    dims = None
    entries = []
    comments = []
    with gzip.open(path, "rt", errors="replace") as handle:
        header = handle.readline().strip()
        for line in handle:
            line = line.strip()
            if not line:
                continue
            if line.startswith("%"):
                if len(comments) < 3:
                    comments.append(line)
                continue
            dims = line
            break
        for line in handle:
            line = line.strip()
            if not line or line.startswith("%"):
                continue
            entries.append(line.split())
            if len(entries) >= n:
                break
    print(f"Matrix Market header: {header}")
    if comments:
        print("Comments:")
        for comment in comments:
            print(f"  {comment}")
    print(f"Matrix dimensions line: {dims}")
    columns = ["row", "column", "value"] if entries and len(entries[0]) >= 3 else None
    show_table(entries, columns=columns)


def preview_h5ad(path, n=HEAD_N):
    if ad is None:
        print("Install/load anndata in this kernel to inspect .h5ad files.")
        if h5py is not None:
            with h5py.File(path, "r") as handle:
                print(f"HDF5 top-level keys: {list(handle.keys())}")
        return

    adata = ad.read_h5ad(path, backed="r")
    try:
        print(adata)
        print(f"n_obs: {adata.n_obs:,}; n_vars: {adata.n_vars:,}")
        print("obs head:")
        display(adata.obs.head(n))
        print("var head:")
        display(adata.var.head(n))
        try:
            x = adata.X[:n, :n]
            if hasattr(x, "toarray"):
                x = x.toarray()
            if pd is not None:
                display(pd.DataFrame(x, index=adata.obs_names[:n], columns=adata.var_names[:n]))
            else:
                print(x)
        except Exception as exc:
            print(f"Could not preview X[:{n}, :{n}]: {exc}")
    finally:
        try:
            adata.file.close()
        except Exception:
            pass


def preview_loom(path, n=HEAD_N):
    if h5py is None:
        print("Install/load h5py or loompy in this kernel to inspect .loom files.")
        return

    with h5py.File(path, "r") as handle:
        print(f"HDF5 top-level keys: {list(handle.keys())}")
        matrix = handle.get("matrix")
        if matrix is None:
            print("No /matrix dataset found in loom file.")
            return
        print(f"matrix shape: {matrix.shape}; dtype: {matrix.dtype}")
        nr = min(n, matrix.shape[0])
        nc = min(n, matrix.shape[1])
        sample = matrix[:nr, :nc]
        if pd is not None:
            display(pd.DataFrame(sample))
        else:
            print(sample)
        row_attrs = list(handle.get("row_attrs", {}).keys()) if "row_attrs" in handle else []
        col_attrs = list(handle.get("col_attrs", {}).keys()) if "col_attrs" in handle else []
        print(f"row_attrs keys: {row_attrs[:20]}")
        print(f"col_attrs keys: {col_attrs[:20]}")


def preview_text(path, n=HEAD_N):
    with open(path, "rt", errors="replace") as handle:
        for idx, line in enumerate(handle):
            if idx >= n:
                break
            print(line.rstrip("\n")[:1000])


def preview_pickle(path):
    try:
        with open(path, "rb") as handle:
            obj = pickle.load(handle)
    except Exception as exc:
        print(f"Could not load pickle preview: {exc}")
        return
    print(f"pickle object type: {type(obj)}")
    if hasattr(obj, "head"):
        display(obj.head(HEAD_N))
    elif isinstance(obj, dict):
        keys = list(obj.keys())[:HEAD_N]
        print(f"first keys: {keys}")
        for key in keys:
            print(f"{key!r}: {repr(obj[key])[:500]}")
    else:
        print(repr(obj)[:1000])


def inspect_dataset(ds):
    if not print_dataset_header(ds):
        return
    path = Path(ds["path"])
    suffixes = "".join(path.suffixes).lower()
    if suffixes.endswith(".h5ad"):
        preview_h5ad(path)
    elif suffixes.endswith(".mtx.gz"):
        preview_mtx_gz(path)
    elif suffixes.endswith(".tsv.gz"):
        preview_delimited_gz(path)
    elif suffixes.endswith(".loom"):
        preview_loom(path)
    elif suffixes.endswith(".txt"):
        preview_text(path)
    elif suffixes.endswith(".pickle"):
        preview_pickle(path)
    else:
        print(f"No preview helper for suffixes: {suffixes}")


## Inspect All Main Datasets

Run the next cell to preview every main dataset. It may still take a few minutes because there are many large files, but it should not load the full 90G into memory.

In [10]:
for ds in DATASETS:
    inspect_dataset(ds)
    print("\n" + "-" * 100 + "\n")


### Construction of a human cell landscape at single-cell level

Source: CellxGene
Path: /gladstone/theodoris/lab/enockniyonkuru/maxtoki_development_data/source_1_cellxgene/01_construction_of_a_human_cell_landscape_at_single_cell_level.h5ad
Origin: https://datasets.cellxgene.cziscience.com/74c3403a-451c-4a62-84e0-d8a8e45c7ea7.h5ad
Size: 1.3GiB
AnnData object with n_obs × n_vars = 599926 × 26069 backed at '/gladstone/theodoris/lab/enockniyonkuru/maxtoki_development_data/source_1_cellxgene/01_construction_of_a_human_cell_landscape_at_single_cell_level.h5ad'
    obs: 'batch', 'n_genes', 'n_counts', 'louvain', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'tissue_original', 'assay_ontology_term_id', 'tissue_ontology_term_id', 'author_cell_type', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'is_primary_data', 'donor_id', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'f

,batch,n_genes,n_counts,louvain,disease_ontology_term_id,self_reported_ethnicity_ontology_term_id,tissue_original,assay_ontology_term_id,tissue_ontology_term_id,author_cell_type,...,suspension_type,tissue_type,cell_type,assay,disease,sex,tissue,self_reported_ethnicity,development_stage,observation_joinid
index,,,,,,,,,,,,,,,,,,,,,
AdultAdipose_1.TATGTAACACCCGCAGGA-0,AdultAdipose_1,911,1285.0,2,PATO:0000461,HANCESTRO:0027,AdultAdipose,EFO:0030002,UBERON:0001013,Macrophage,...,cell,tissue,macrophage,microwell-seq,normal,male,adipose tissue,Han Chinese,36-year-old stage,P!O5w%5Yll
AdultAdipose_1.CTCGCAAATAAACATCCC-0,AdultAdipose_1,1075,1628.0,2,PATO:0000461,HANCESTRO:0027,AdultAdipose,EFO:0030002,UBERON:0001013,Macrophage,...,cell,tissue,macrophage,microwell-seq,normal,male,adipose tissue,Han Chinese,36-year-old stage,A7^kr&W!s$
AdultAdipose_1.CGAGTATCGTAATACTTC-0,AdultAdipose_1,730,1018.0,2,PATO:0000461,HANCESTRO:0027,AdultAdipose,EFO:0030002,UBERON:0001013,Macrophage,...,cell,tissue,macrophage,microwell-seq,normal,male,adipose tissue,Han Chinese,36-year-old stage,vgUBNp?*1%
AdultAdipose_1.CTCGCAGCGAATCGGCAG-0,AdultAdipose_1,802,1145.0,2,PATO:0000461,HANCESTRO:0027,AdultAdipose,EFO:0030002,UBERON:0001013,Macrophage,...,cell,tissue,macrophage,microwell-seq,normal,male,adipose tissue,Han Chinese,36-year-old stage,jeH|zunzv0
AdultAdipose_1.ACGTTGATCAACAGATGG-0,AdultAdipose_1,622,852.0,2,PATO:0000461,HANCESTRO:0027,AdultAdipose,EFO:0030002,UBERON:0001013,Macrophage,...,cell,tissue,macrophage,microwell-seq,normal,male,adipose tissue,Han Chinese,36-year-old stage,Vts1QZY%G(


var head:


,feature_is_filtered,feature_name,feature_reference,feature_biotype,feature_length,feature_type
ENSG00000185189,False,NRBP2,NCBITaxon:9606,gene,2164,protein_coding
ENSG00000124191,False,TOX2,NCBITaxon:9606,gene,2376,protein_coding
ENSG00000116791,False,CRYZ,NCBITaxon:9606,gene,1592,protein_coding
ENSG00000271993,False,ENSG00000271993,NCBITaxon:9606,gene,628,lncRNA
ENSG00000273110,False,ENSG00000273110,NCBITaxon:9606,gene,681,lncRNA


...

row_attrs keys: ['ensembl_id', 'ensembl_id_version', 'ensembl_original', 'gene_name', 'gene_original', 'gene_type']
col_attrs keys: ['cell_id', 'dataset', 'filter_pass', 'n_counts', 'n_func_genes', 'organ_specific', 'pct_mito', 'platform', 'sample', 'study', 'unique_cell_id']

----------------------------------------------------------------------------------------------------



### Fetal vs. adult human epicardium

Source: Lab Directory
Path: /gladstone/theodoris/lab/enockniyonkuru/maxtoki_development_data/source_4_lab_directory/04_fetal_vs_adult_human_epicardium.loom
Origin: /gladstone/theodoris/lab/genecorpus_XM/corpus_loom_files/genecorpus_95M_loom/cxg/5500c673-1610-40a0-86d9-64d987ae50e6.loom
Size: 302.0MiB
HDF5 top-level keys: ['attrs', 'col_attrs', 'col_graphs', 'layers', 'matrix', 'row_attrs', 'row_graphs']
matrix shape: (60664, 30889); dtype: float32


,0,1,2,3,4
0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0


row_attrs keys: ['ensembl_id', 'ensembl_id_version', 'ensembl_original', 'gene_name', 'gene_type']
col_attrs keys: ['cell_subclass', 'cell_type', 'dataset', 'development_stage', 'disease', 'donor_id', 'filter_pass', 'n_counts', 'n_func_genes', 'organ_specific', 'pct_mito', 'platform', 'sample', 'self_reported_ethnicity', 'sex', 'study', 'tissue', 'tissue_general', 'unique_cell_id']

----------------------------------------------------------------------------------------------------



## Optional: Inspect One Source at a Time

Use this section when you want smaller output chunks.

In [5]:
available_sources = sorted({ds["source"] for ds in DATASETS})
available_sources


['CellxGene', 'Human Early Embryogenesis Atlas', 'Lab Directory', 'UCSC Cells']

In [9]:
SOURCE_TO_RUN = "Lab Directory"  # Change to one of available_sources.

for ds in DATASETS:
    if ds["source"] == SOURCE_TO_RUN:
        inspect_dataset(ds)
        print("\n" + "-" * 100 + "\n")


### Human fetal cis-regulatory elements

Source: Lab Directory
Path: /gladstone/theodoris/lab/enockniyonkuru/maxtoki_development_data/source_4_lab_directory/01_human_fetal_cis_regulatory_elements.loom
Origin: /gladstone/theodoris/lab/genecorpus_XM/corpus_loom_files/genecorpus_95M_loom/9010519993/File_S3_Matrix.Gene_Raw_Counts_of_Cells.loom
Size: 1.2GiB
HDF5 top-level keys: ['attrs', 'col_attrs', 'col_graphs', 'layers', 'matrix', 'row_attrs', 'row_graphs']
matrix shape: (19329, 185061); dtype: float64


,0,1,2,3,4
0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,1.0


row_attrs keys: ['ensembl_id', 'ensembl_id_version', 'ensembl_original', 'gene_name', 'gene_original', 'gene_type']
col_attrs keys: ['cell_id', 'dataset', 'filter_pass', 'n_counts', 'n_func_genes', 'organ_specific', 'pct_mito', 'platform', 'sample', 'study', 'unique_cell_id']

----------------------------------------------------------------------------------------------------



### Human fetal striatum atlas

...

row_attrs keys: ['ensembl_id', 'ensembl_id_version', 'ensembl_original', 'gene_name', 'gene_original', 'gene_type']
col_attrs keys: ['cell_id', 'dataset', 'filter_pass', 'n_counts', 'n_func_genes', 'organ_specific', 'pct_mito', 'platform', 'sample', 'study', 'unique_cell_id']

----------------------------------------------------------------------------------------------------



### Fetal vs. adult human epicardium

Source: Lab Directory
Path: /gladstone/theodoris/lab/enockniyonkuru/maxtoki_development_data/source_4_lab_directory/04_fetal_vs_adult_human_epicardium.loom
Origin: /gladstone/theodoris/lab/genecorpus_XM/corpus_loom_files/genecorpus_95M_loom/cxg/5500c673-1610-40a0-86d9-64d987ae50e6.loom
Size: 302.0MiB
HDF5 top-level keys: ['attrs', 'col_attrs', 'col_graphs', 'layers', 'matrix', 'row_attrs', 'row_graphs']
matrix shape: (60664, 30889); dtype: float32


,0,1,2,3,4
0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0


row_attrs keys: ['ensembl_id', 'ensembl_id_version', 'ensembl_original', 'gene_name', 'gene_type']
col_attrs keys: ['cell_subclass', 'cell_type', 'dataset', 'development_stage', 'disease', 'donor_id', 'filter_pass', 'n_counts', 'n_func_genes', 'organ_specific', 'pct_mito', 'platform', 'sample', 'self_reported_ethnicity', 'sex', 'study', 'tissue', 'tissue_general', 'unique_cell_id']

----------------------------------------------------------------------------------------------------



## Optional: Supporting Files

These files are not primary datasets, but they are useful QC/log artifacts copied with the megakaryocyte development loom files.

In [ ]:
for path in SUPPORTING_FILES:
    display(Markdown(f"### {path.name}"))
    print(f"Path: {path}")
    if not path.exists():
        print("MISSING")
        continue
    print(f"Size: {format_bytes(path.stat().st_size)}")
    suffixes = "".join(path.suffixes).lower()
    if suffixes.endswith(".txt"):
        preview_text(path)
    elif suffixes.endswith(".pickle"):
        preview_pickle(path)
    print("\n" + "-" * 100 + "\n")
